<a href="https://colab.research.google.com/github/snehapathak9/Summer-Analytics/blob/main/Planet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

In [2]:
# Load data
train = pd.read_csv('/content/Train_Data.csv')
test = pd.read_csv('/content/Test_Data.csv')

In [3]:
# Preprocessing function
def preprocess_data(train_df, test_df):
    # Handle missing targets
    train_df = train_df.dropna(subset=['age_group'])

    # Basic mappings
    train_df['RIAGENDR'] = train_df['RIAGENDR'].map({1: 0, 2: 1})
    test_df['RIAGENDR'] = test_df['RIAGENDR'].map({1: 0, 2: 1})

    train_df['PAQ605'] = train_df['PAQ605'].map({1: 1, 2: 0})
    test_df['PAQ605'] = test_df['PAQ605'].map({1: 1, 2: 0})

    # Convert target
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    # Original features only
    features = ['RIAGENDR', 'PAQ605', 'BMXBMI', 'LBXGLU', 'DIQ010', 'LBXGLT', 'LBXIN']

    # Prepare data
    X = train_df[features]
    y = train_df['age_group']
    X_test = test_df[features]

    return X, y, X_test

In [4]:
# Prepare data
X, y, X_test = preprocess_data(train, test)

/tmp/ipython-input-3-2066097319.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['RIAGENDR'] = train_df['RIAGENDR'].map({1: 0, 2: 1})
/tmp/ipython-input-3-2066097319.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['PAQ605'] = train_df['PAQ605'].map({1: 1, 2: 0})
/tmp/ipython-input-3-2066097319.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentatio

In [5]:
# Handle missing values
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_test_imputed = imputer.transform(X_test)

In [6]:
# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [7]:
# Split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
# Optimized Random Forest
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [9]:
# Train model
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=5,
                       min_samples_split=10, n_estimators=500, n_jobs=-1,
                       random_state=42)

In [10]:
# Validate
val_pred = model.predict(X_val)
val_f1 = f1_score(y_val, val_pred)
print(f"Validation F1: {val_f1:.4f}")

Validation F1: 0.2968


In [11]:
val_probs = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.2, 0.8, 101)
best_threshold = 0.5
best_f1 = 0

for thresh in thresholds:
    adjusted_pred = (val_probs >= thresh).astype(int)
    f1 = f1_score(y_val, adjusted_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

print(f"Optimized Threshold: {best_threshold:.4f}")
print(f"Optimized Validation F1: {best_f1:.4f}")

Optimized Threshold: 0.3800
Optimized Validation F1: 0.3942


In [12]:
# Train final model on all data
model.fit(X_scaled, y)


RandomForestClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=5,
                       min_samples_split=10, n_estimators=500, n_jobs=-1,
                       random_state=42)

In [13]:

# Predict test set
test_probs = model.predict_proba(X_test_scaled)[:, 1]
test_pred = (test_probs >= best_threshold).astype(int)

In [14]:
# Create submission
submission = pd.DataFrame({'age_group': test_pred})
submission.to_csv('optimized_submission.csv', index=False)
print("Submission file created")

Submission file created


In [15]:
# Analysis
print("\nPrediction distribution:")
print(f"Adults (0): {sum(test_pred == 0)}")
print(f"Seniors (1): {sum(test_pred == 1)}")
print(f"Validation F1 was: {best_f1:.4f}")


Prediction distribution:
Adults (0): 190
Seniors (1): 122
Validation F1 was: 0.3942


In [16]:
from google.colab import files
files.download('optimized_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>